# Week 4.1 — Segmentation with Clustering

**Course:** Data Science for Business (YSU)  
**Prerequisite:** [Week 4 — RFM](./Week_4.Basics_of_Segmentation_RFM.ipynb)  
**Next:** [Week 4.2 — PCA + Clustering](./Week_4.2_Clustering_with_PCA.ipynb)

---

### Learning objectives

1. Build an RFM feature table for clustering.
2. Run **hierarchical clustering** and read a dendrogram.
3. Choose **K** with the elbow method and fit **K-Means**.
4. Profile clusters and name segments for business use.

### Agenda (~45 min)

| Block | Method |
|---|---|
| 1 | RFM features + scaling |
| 2 | Hierarchical clustering |
| 3 | K-Means on RFM |
| 4 | K-Means with extra features |


## 1. Load data and build RFM features

We reuse the same retail file. Clustering needs numeric columns only — start with the three RFM metrics from the previous notebook.

> **Note:** here **monetary** is the **average order value** (mean spend per order date), not total lifetime spend. That matches the original lecture and keeps scales comparable when we add more features later.


In [ ]:
from pathlib import Path
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline


def _acq_weights(n):
    w = np.linspace(1.4, 0.7, n)
    w[0] *= 1.6
    return w / w.sum()


def make_synthetic_retail(n_customers=1800, seed=5) -> pd.DataFrame:
    """Fallback data when data/data_cleared.csv is missing."""
    rng = np.random.default_rng(seed)
    months = pd.period_range("2010-12", "2011-12", freq="M")
    first = rng.choice(months[:-1], size=n_customers, p=_acq_weights(len(months) - 1))
    rows = []
    invoice = 10000
    for i, start in enumerate(first):
        cid = 10000 + i
        quality = rng.uniform(0.15, 0.55)
        first_spend = float(rng.lognormal(3.4, 0.7))
        for m in months[months >= start]:
            age = (m - start).n
            if age == 0:
                p_buy = 1.0
            else:
                p_buy = quality * (0.72 ** max(age - 1, 0))
                p_buy *= 1.25 if m.month == 12 else 1.0
                p_buy = min(p_buy, 0.95)
            if rng.random() > p_buy:
                continue
            n_lines = int(rng.integers(1, 5))
            spend = first_spend if age == 0 else first_spend * rng.uniform(0.4, 1.3)
            for _ in range(n_lines):
                invoice += 1
                qty = int(rng.integers(1, 8))
                rows.append(
                    {
                        "InvoiceNo": invoice,
                        "InvoiceDate": m.to_timestamp()
                        + pd.Timedelta(days=int(rng.integers(0, 27))),
                        "CustomerID": cid,
                        "Quantity": qty,
                        "TotalPrice": spend / n_lines,
                    }
                )
    return pd.DataFrame(rows)


def load_transactions() -> pd.DataFrame:
    for path in [Path("data/data_cleared.csv"), Path("../data/data_cleared.csv")]:
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {len(df):,} line items from {path.resolve()}")
            return df
    print("data/data_cleared.csv not found — using synthetic retail data.")
    return make_synthetic_retail()


data = load_transactions()
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
data["CustomerID"] = pd.to_numeric(data["CustomerID"], errors="coerce")
data = data.dropna(subset=["CustomerID", "InvoiceDate"]).copy()
data["CustomerID"] = data["CustomerID"].astype("int64")
print(
    f"{data['CustomerID'].nunique():,} customers | "
    f"{data['InvoiceDate'].min().date()} to {data['InvoiceDate'].max().date()}"
)
data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans

In [ ]:
snapshot = data["InvoiceDate"].max()

dt = data.groupby(["CustomerID", "InvoiceDate"], as_index=False)["TotalPrice"].sum()

rfm = (
    dt.groupby("CustomerID")
    .agg(
        recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
        frequency=("InvoiceDate", "count"),
        monetary=("TotalPrice", "mean"),
    )
    .reset_index()
)

rfm.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rfm.drop(columns="CustomerID").corr(), annot=True, cmap="RdBu", vmin=-1, vmax=1, ax=ax)
ax.set_title("RFM correlation heatmap")
plt.tight_layout()
plt.show()

## 2. Scale features

K-Means uses distance. **Always standardize** continuous features first (zero mean, unit variance).


In [ ]:
cluster_data = rfm.drop(columns="CustomerID")
scaler = StandardScaler()
data_stand = scaler.fit_transform(cluster_data)

pd.DataFrame(data_stand, columns=cluster_data.columns).describe().round(2)

## 3. Hierarchical clustering

**Idea:** start with each customer as their own cluster, repeatedly merge the two closest clusters.

The **dendrogram** shows merge history. The **Ward** method minimizes within-cluster variance when merging.

We truncate the tree for readability — your dataset has thousands of leaves.


In [ ]:
hier_clust = linkage(data_stand, method="ward")

fig, ax = plt.subplots(figsize=(12, 6))
dendrogram(hier_clust, truncate_mode="level", p=10, no_labels=True, ax=ax)
ax.set_title("Hierarchical clustering dendrogram (truncated)")
ax.set_xlabel("Observations")
ax.set_ylabel("Distance")
plt.tight_layout()
plt.show()

The dendrogram suggests **2–4** large groups, but hierarchical clustering does not scale well to huge datasets. It is useful for intuition; **K-Means** is the workhorse in industry.


## 4. K-Means on RFM

**Elbow method:** plot within-cluster sum of squares (WCSS / inertia) vs number of clusters $K$. Pick $K$ where the curve bends.


In [ ]:
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(data_stand)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 11), wcss, marker="o", linestyle="--")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("WCSS (inertia)")
ax.set_title("Elbow plot — K-Means on RFM")
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, init="k-means++", random_state=42, n_init="auto")
kmeans.fit(data_stand)

segm_labels = cluster_data.copy()
segm_labels["Segments"] = kmeans.labels_
segm_labels.head()

### Profile and name clusters

Group by cluster label and inspect **mean R, F, M**. Then assign human-readable names.


In [ ]:
profiling = segm_labels.groupby("Segments", as_index=False).mean()
profiling["Segment_size"] = segm_labels.groupby("Segments").size().values
profiling["Segment_prop"] = (profiling["Segment_size"] / profiling["Segment_size"].sum() * 100).round(1)

name_map = {0: "promising", 1: "champions", 2: "lost", 3: "high spenders"}
profiling["Segments"] = profiling["Segments"].map(name_map)
segm_labels["Segments"] = segm_labels["Segments"].map(name_map)
profiling

### Visualize clusters in 2D

Pick two RFM axes at a time. Clusters should separate along recency and frequency most clearly.


In [ ]:
def plot_segments(x_col, y_col, title):
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.scatterplot(data=segm_labels, x=x_col, y=y_col, hue="Segments", palette="Set2", alpha=0.6, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_segments("monetary", "frequency", "K-Means segments: Monetary vs Frequency")
plot_segments("monetary", "recency", "K-Means segments: Monetary vs Recency")
plot_segments("frequency", "recency", "K-Means segments: Frequency vs Recency")

## 5. Add more features

RFM is a good start, but we can enrich the customer profile:

| Feature | Meaning |
|---|---|
| `AvgQuantity` | average items per order |
| `AvgDifferentProducts` | average line items per order |
| `GapBetweenOrders` | days between first and last order |
| `Monetary_Value` | mean order value |


In [ ]:
dt_ext = data.groupby(["CustomerID", "InvoiceDate"], as_index=False).agg(
    TotalPrice=("TotalPrice", "sum"),
    Quantity=("Quantity", "sum"),
    InvoiceNo=("InvoiceNo", "count"),
)

customer_data = dt_ext.groupby("CustomerID").agg(
    AvgQuantity=("Quantity", "mean"),
    AvgDifferentProducts=("InvoiceNo", "mean"),
    Recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    Frequency=("CustomerID", "count"),
    Monetary_Value=("TotalPrice", "mean"),
    GapBetweenOrders=("InvoiceDate", lambda d: (d.max() - d.min()).days),
)

customer_data.head()

In [ ]:
data_stand_ext = scaler.fit_transform(customer_data)

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    km.fit(data_stand_ext)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, 11), wcss, marker="o", linestyle="--")
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("WCSS")
ax.set_title("Elbow plot — extended feature set")
plt.tight_layout()
plt.show()

In [ ]:
kmeans_ext = KMeans(n_clusters=3, random_state=42, n_init="auto")
kmeans_ext.fit(data_stand_ext)

segm_ext = customer_data.copy()
segm_ext["Segments"] = kmeans_ext.labels_

prof_ext = segm_ext.groupby("Segments", as_index=False).mean()
prof_ext["Segment_size"] = segm_ext.groupby("Segments").size().values
prof_ext["Segment_prop"] = (prof_ext["Segment_size"] / prof_ext["Segment_size"].sum() * 100).round(1)

ext_names = {0: "lost", 1: "promising", 2: "champions"}
prof_ext["Segments"] = prof_ext["Segments"].map(ext_names)
segm_ext["Segments"] = segm_ext["Segments"].map(ext_names)
prof_ext

In [ ]:
plot_df = segm_ext.reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [
    ("Monetary_Value", "AvgQuantity"),
    ("Recency", "Frequency"),
    ("GapBetweenOrders", "Frequency"),
]
for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(data=plot_df, x=x, y=y, hue="Segments", palette="Set2", alpha=0.55, ax=ax)
    ax.set_title(f"{y} vs {x}")
plt.tight_layout()
plt.show()

## 6. Wrap-up

- **RFM rules** (previous notebook) are transparent; **K-Means** finds data-driven groups.
- Always **scale** before distance-based methods.
- Cluster labels are meaningless until you **profile means** and assign names.
- Next: [Week 4.2](./Week_4.2_Clustering_with_PCA.ipynb) applies PCA before K-Means on the extended feature set.

### Practice

1. Try `K=5` on RFM only. Do the segment names still make sense?
2. Compute the **silhouette score** for K = 2..6 on `data_stand`.
3. Which segment would you target for a reactivation email?
